# Sequential Workflow: Prompt Chaining

`START -> create_outline -> create_blog -> END`

The output of the first LLM call becomes the input of the second.

In [1]:
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from typing import TypedDict
from dotenv import load_dotenv

load_dotenv()

model = ChatGroq(model='openai/gpt-oss-20b')

In [2]:
class BlogState(TypedDict):
    title: str
    outline: str
    content: str

In [3]:
def create_outline(state: BlogState) -> BlogState:
    prompt = f"Generate a short outline (4-5 bullet points) for a blog on: {state['title']}"
    state['outline'] = model.invoke(prompt).content
    return state


def create_blog(state: BlogState) -> BlogState:
    prompt = f"Write a short blog (about 150 words) titled '{state['title']}' using this outline:\n{state['outline']}"
    state['content'] = model.invoke(prompt).content
    return state

In [4]:
graph = StateGraph(BlogState)

graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)

graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', END)

workflow = graph.compile()

In [5]:
result = workflow.invoke({'title': 'Rise of AI in India'})
print(result['outline'])
print('-' * 50)
print(result['content'])

**Outline: “Rise of AI in India”**

- **Historical Context & Current Landscape**  
  - Quick timeline of AI milestones in India (research, startups, academic breakthroughs).  

- **Government & Policy Initiatives**  
  - Digital India, National AI Strategy, funding schemes, and public‑private partnerships.  

- **Industry Adoption & Success Stories**  
  - Key sectors (healthcare, finance, agriculture, manufacturing) and notable Indian AI startups.  

- **Talent Development & Ecosystem Growth**  
  - Universities, bootcamps, AI research hubs, and the role of diaspora talent.  

- **Challenges & Future Outlook**  
  - Data privacy, infrastructure gaps, ethical considerations, and projected growth trajectories.
--------------------------------------------------
**Rise of AI in India**

**Historical Context & Current Landscape** – AI in India began with early research at institutions like IITs in the 1990s, followed by the launch of the “AI for All” initiative in 2015. The 2017‑18 period 